# Talent Marketplace — Data Analysis
**Portfolio project | Carlos Hisamitsu**

End-to-end analysis of a synthetic HR dataset modelled on a real internal talent marketplace:
vendor data ingestion → data quality → workforce analytics → career impact of gig participation.

**Dataset:** 5,000 employees · 1,500 gigs · 3-year window (2023–2025) · 30 countries  
**Data source:** `data/dirty/` — realistic data quality issues injected (see `injection_manifest.csv`)

---


## 1. Setup

In [246]:
# Uncomment to install if needed
# %pip install pandas numpy matplotlib seaborn scipy statsmodels pyarrow openpyxl

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import statsmodels.api as sm

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

RAW_DIR    = Path('../data/dirty')
EXPORT_DIR = Path('../data/export')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

LEVELS = ['L1','L2','L3','L4','L5','L6','L7']
REF_DATE = pd.Timestamp('2025-12-31')

print('Ready.')
print(f'Reading from: {RAW_DIR.resolve()}')


Ready.
Reading from: C:\development\data-analytics\hr-analytics-pipeline\data\dirty


## 2. Data Audit

Before transforming anything, profile every file.  
Goal: understand shape, nulls, and duplicates — and flag anything unexpected.


In [247]:
FILES = {
    'employee_master'     : ('raw_employee_master.csv',
                             ['birth_date','hire_date','termination_date']),
    'job_assignment'      : ('raw_employee_job_assignment.csv',
                             ['start_date','end_date']),
    'skill_dim'           : ('dim_skill.csv', []),
    'employee_skills'     : ('raw_employee_skills.csv', ['added_date']),
    'gig_master'          : ('raw_gig_master.csv', ['posted_date']),
    'gig_req_skills'      : ('raw_gig_required_skills.csv', []),
    'applications'        : ('raw_gig_applications_and_assignments.csv',
                             ['application_date','manager_approval_date',
                              'assignment_start_date','assignment_end_date']),
    'activity_log'        : ('raw_user_activity_log.csv', ['event_timestamp']),
    'training_master'     : ('raw_training_master.csv', []),
    'training_skills'     : ('raw_training_skills.csv', []),
    'training_records'    : ('raw_training_records.csv', ['completion_date']),
}

raw = {}
audit = []

for key, (fname, date_cols) in FILES.items():
    df = pd.read_csv(RAW_DIR / fname,
                     parse_dates=date_cols if date_cols else False,
                     low_memory=False)
    raw[key] = df

    null_cols = (df.isnull().sum() / len(df) * 100).round(1)
    null_cols = null_cols[null_cols > 0].sort_values(ascending=False)

    if not null_cols.empty: # null_cols:
        worst_null_col = null_cols.idxmax()
        worst_null_pct = null_cols.max()
        worst_null_list = ", ".join(f"{col}: {round(pct)}%" for col, pct in null_cols.items())
    else:
        worst_null_col = None
        worst_null_pct = 0
        worst_null_list = ""

    audit.append({
        'table'             : key,
        'rows'              : len(df),
        'duplicates'        : int(df.duplicated().sum()),
        'columns'           : df.shape[1],
        'null_cols'         : len(null_cols),
        'worst_null_col'    : worst_null_col,
        'worst_null%'       : round(worst_null_pct, 1),
        "null_list"         : worst_null_list
    })

audit_df = pd.DataFrame(audit)
print(audit_df.to_string(index=False))


           table   rows  duplicates  columns  null_cols         worst_null_col  worst_null%                                                                                                      null_list
 employee_master   5000           0       12          2       termination_date        73.10                                                                   termination_date: 73%, employment_status: 1%
  job_assignment   8678           0       12          2               end_date        59.20                                                                                  end_date: 59%, manager_id: 5%
       skill_dim     80           0        3          0                    NaN         0.00                                                                                                               
 employee_skills  94490        2204        5          1           skill_source         2.90                                                                                               sk

**What I'm looking for here:**
- Duplicates
- Unexpected nulls in key columns (termination dates, manager IDs)
- Row counts consistent with what the source system should export

Issues found will be addressed one by one in Section 3.


## 3. Data Quality

Real data is never clean — especially when it comes from vendor SFTP exports,
HR systems with optional fields, or LMS webhooks that retry on failure.

For each table I run targeted checks against the actual data.
Each check produces a count of affected rows and a sample for inspection.
Issues found are then fixed with a documented, conservative transform.

The approach: **Detect → Inspect → Fix → Verify**


### 3.1 Employee master

In [248]:
emp = raw['employee_master'].copy()
issues = {}

# ── Check 1: Terminated employees missing termination_date ───────────────────
# Logic: status = Terminated implies a date exists. Null means the HR admin
# updated the employment status but skipped the date field.
mask_term_null = (emp['employment_status'].isin(['Terminated', 'Retired'])) & emp['termination_date'].isna()
issues['Terminated or Retired with no termination_date'] = int(mask_term_null.sum())
if mask_term_null.any():
    print(f"[FOUND] Terminated or Retired with no termination_date: {mask_term_null.sum()} rows")
    print(emp[mask_term_null][['employee_id','hire_date','employment_status','termination_date']].head(3).to_string(index=False))
    print()

# Fix: impute using median tenure of employees who do have a date
median_days = int(
    (emp.dropna(subset=['termination_date'])['termination_date'] -
     emp.dropna(subset=['termination_date'])['hire_date']).dt.days.median()
)

emp.loc[mask_term_null, 'termination_date'] = (
    emp.loc[mask_term_null, 'hire_date'] + pd.Timedelta(days=median_days)
)
emp['term_date_imputed'] = mask_term_null
print(f"  Fix: imputed as hire_date + {median_days} days (median tenure of terminated employees)")
print(f"  Verify: nulls remaining = {emp[emp['employment_status'].isin(['Terminated', 'Retired'])]['termination_date'].isna().sum()}")


[FOUND] Terminated or Retired with no termination_date: 49 rows
 employee_id  hire_date employment_status termination_date
EMP-97F475F5 2023-12-06        Terminated              NaT
EMP-70A3D9D2 2022-02-25        Terminated              NaT
EMP-D848706B 2018-07-13        Terminated              NaT

  Fix: imputed as hire_date + 923 days (median tenure of terminated employees)
  Verify: nulls remaining = 0


In [249]:
# ── Check 2: Future hire dates ───────────────────────────────────────────────
# Logic: hire_date cannot be after our analysis date.
# Cause: HR pre-loads upcoming starters before their actual start date.
mask_future = emp['hire_date'] > REF_DATE
issues['Future hire_date'] = int(mask_future.sum())
if mask_future.any():
    print(f"[FOUND] Future hire_date: {mask_future.sum()} rows out of {len(emp)} records")
    print(emp[mask_future][['employee_id','hire_date','employment_status']].head(3).to_string(index=False))
    print()

emp = emp[~mask_future].copy()
print(f"  Fix: drop future hired employees after {REF_DATE.date()}")
print(f"  Verify: future dates remaining = {(emp['hire_date'] > REF_DATE).sum()} / employee records remaining: {len(emp)}")


[FOUND] Future hire_date: 63 rows out of 5000 records
 employee_id  hire_date employment_status
EMP-045C2852 2026-05-18            Active
EMP-AD801586 2026-03-01            Active
EMP-CACD9FB1 2026-05-16            Active

  Fix: drop future hired employees after 2025-12-31
  Verify: future dates remaining = 0 / employee records remaining: 4937


In [250]:
# ── Check 3: Missing employment_status ───────────────────────────────────────
# Logic: every employee needs a status. Nulls suggest a middleware integration
# that silently dropped the field on a partial API response.
mask_no_status = emp['employment_status'].isna() | (emp['employment_status'] == '')
issues['Null employment_status'] = int(mask_no_status.sum())
if mask_no_status.any():
    print(f"[FOUND] Null employment_status: {mask_no_status.sum()} rows")
    print(emp[mask_no_status][['employee_id','hire_date','termination_date']].head(3).to_string(index=False))
    print()

emp.loc[mask_no_status & emp['termination_date'].notna(), 'employment_status'] = 'Terminated'
emp.loc[mask_no_status & emp['termination_date'].isna(),  'employment_status'] = 'Active'

emp['employment_status_imputed'] = mask_no_status
print(f"  Fix: derived from termination_date for {mask_no_status.sum()} rows")
print(f"  Verify: nulls remaining = {emp['employment_status'].isna().sum()}")


[FOUND] Null employment_status: 49 rows
 employee_id  hire_date termination_date
EMP-112B1E91 2019-11-08              NaT
EMP-EB9C11B2 2020-03-01              NaT
EMP-A7DEBCDE 2019-03-17       2024-08-09

  Fix: derived from termination_date for 49 rows
  Verify: nulls remaining = 0


In [251]:
# ── Check 4: Duplicate email addresses ───────────────────────────────────────
# Logic: company_email must be unique. Duplicates appear when an employee is re-hired
# with same old email or another employee with same name of a former employee is hired.
# (duplicated active emails are blocked at the AD creation, so we will not consider this case)
mask_dup = emp.duplicated(subset=['company_email'], keep=False)
issues['Duplicate company_email'] = int(mask_dup.sum())
if mask_dup.any():
     print(f"\n[FOUND] Duplicate emails: {mask_dup.sum()} rows across {emp[mask_dup]['company_email'].nunique()} addresses")
     print(emp[mask_dup].groupby('company_email').head(2)[['employee_id','first_name','last_name','company_email']].head(4).to_string(index=False))
     print()

emp_term = emp['employment_status'].isin(['Terminated', 'Retired'])
emp.loc[emp_term, 'company_email'] = (
    emp.loc[emp_term, 'employee_id'].astype(str) + '@' + 
    emp.loc[emp_term, 'company_email'].str.split('@').str[1]
)
print("  Fix: use employee_id for terminated or retired employees' emails")

still_dup = emp.duplicated(subset=['company_email'], keep=False)
issues['Email replaced (data error)'] = int(still_dup.sum())
if still_dup.any():
     print(f"\n[FOUND] There are still duplicate emails: {still_dup.sum()} rows across {emp[still_dup]['company_email'].nunique()} addresses")
     print(emp[still_dup].groupby('company_email').head(2)[['employee_id','first_name','last_name','company_email']].head(4).to_string(index=False))
     print()
     emp.loc[still_dup, 'company_email'] = (
          emp.loc[still_dup, 'employee_id'].astype(str) + '@' + 
          emp.loc[still_dup, 'company_email'].str.split('@').str[1]
          )
     
emp['duplicated_email_imputed'] = still_dup
print("  Fix: use employee_id for emails that are still duplicated after fixing terminated/retired employees")
print(f"  Verify: duplicates remaining = {emp.duplicated(subset=['company_email']).sum()}")



[FOUND] Duplicate emails: 143 rows across 71 addresses
 employee_id first_name last_name                company_email
EMP-EA3C2C53    William  Morrison william.morrison@example.com
EMP-E559C838      Piotr   Janssen  tyler.jackson.2@example.com
EMP-995D8217     Sakura       Cho       sakura.cho@example.com
EMP-58A7EF8C       Riya      Yang        riya.yang@example.com

  Fix: use employee_id for terminated or retired employees' emails

[FOUND] There are still duplicate emails: 62 rows across 31 addresses
 employee_id first_name last_name                company_email
EMP-EA3C2C53    William  Morrison william.morrison@example.com
EMP-58A7EF8C       Riya      Yang        riya.yang@example.com
EMP-B5C8799D  Sebastian   de Boer   elena.muller.1@example.com
EMP-BDE81C98       Erik     Pérez    raj.kobayashi@example.com

  Fix: use employee_id for emails that are still duplicated after fixing terminated/retired employees
  Verify: duplicates remaining = 0


In [252]:
# ── Check 5: Name encoding anomalies ─────────────────────────────────────────
# Logic: flag names with characters suggesting a UTF-8/Latin-1 encoding mismatch.
# Common in vendor SFTP exports where the server locale is misconfigured.
# We flag only — auto-correcting names without a reference list risks data loss.
emp['name_encoding_flag'] = (
    emp['first_name'].apply(lambda x: isinstance(x, str) and not x.isascii()) |
    emp['last_name'].apply(lambda x: isinstance(x, str) and not x.isascii())
)
issues['Name encoding anomalies (flagged)'] = int(emp['name_encoding_flag'].sum())
if emp['name_encoding_flag'].any():
    print(f"\n[FOUND] Encoding anomalies: {emp['name_encoding_flag'].sum()} rows — flagged for manual review")
    print(emp[emp['name_encoding_flag']][['employee_id','first_name','last_name']].head(4).to_string(index=False))



[FOUND] Encoding anomalies: 675 rows — flagged for manual review
 employee_id first_name last_name
EMP-3F550E9D      Clara    Müller
EMP-2E7CE6E7     Markus    Müller
EMP-18DC524D       Hans    Müller
EMP-871F3572      Clara     Pérez


In [253]:
print("\n=== Employee master — summary ===")
for k, v in issues.items():
    print(f"  {'FOUND':<6s} | {k:<47s} | {v:>5,} rows")



=== Employee master — summary ===
  FOUND  | Terminated or Retired with no termination_date  |    49 rows
  FOUND  | Future hire_date                                |    63 rows
  FOUND  | Null employment_status                          |    49 rows
  FOUND  | Duplicate company_email                         |   143 rows
  FOUND  | Email replaced (data error)                     |    62 rows
  FOUND  | Name encoding anomalies (flagged)               |   675 rows


### 3.2 Job assignments (SCD2)

In [254]:
ja = raw['job_assignment'].copy()
ja['job_level'] = pd.Categorical(ja['job_level'], categories=LEVELS, ordered=True)
ja_issues = {}

# ── Check 1: Employee is their own manager ───────────────────────────────────
# Logic: manager_id should never equal employee_id.
# Cause: SuccessFactors/HRIS-HCM sets manager_id = employee_id for org chart root nodes
# (e.g. CEO, country GMs) when no superior position exists in the system.
mask_self = ja['manager_id'] == ja['employee_id']
ja_issues['Self-referencing manager'] = int(mask_self.sum())
if mask_self.any():
    print(f"[FOUND] Self-referencing manager_id: {mask_self.sum()} rows")
    print(ja[mask_self][['employee_id','manager_id','job_level','function']].head(3).to_string(index=False))
    print()
ja.loc[mask_self, 'manager_id'] = None
print(f"  Fix: set to null — no manager is more accurate than a self-reference")

# ── Check 2: Missing manager_id ───────────────────────────────────────────────
# Logic: most employees should have a manager. High null rates typically
# indicate matrix org relationships not mapped in SuccessFactors/HRIS-HCM position management.
mask_no_mgr = ja['manager_id'].isna() | (ja['manager_id'] == '')
ja_issues['Missing manager_id'] = int(mask_no_mgr.sum())
print(f"\n[FOUND] Missing manager_id: {mask_no_mgr.sum()} rows ({mask_no_mgr.mean()*100:.1f}% of assignments)")
ja['manager_missing'] = mask_no_mgr
print(f"  Action: flagged — cannot impute manager without org chart data")


[FOUND] Self-referencing manager_id: 42 rows
 employee_id   manager_id job_level          function
EMP-84578077 EMP-84578077        L5 Quality Assurance
EMP-7F22197E EMP-7F22197E        L4         Marketing
EMP-A755C93C EMP-A755C93C        L5             Sales

  Fix: set to null — no manager is more accurate than a self-reference

[FOUND] Missing manager_id: 475 rows (5.5% of assignments)
  Action: flagged — cannot impute manager without org chart data


In [255]:
# ── Check 3: Overlapping assignment periods ───────────────────────────────────
# Logic: consecutive assignment periods for the same employee must not overlap.
# Cause: two records created during a system migration, both marked active.
ja_tmp = ja.sort_values(['employee_id','start_date'])
ja_tmp['prev_end'] = pd.to_datetime(
    ja_tmp.groupby('employee_id')['end_date'].shift(1), errors='coerce')
overlap_mask = (ja_tmp['prev_end'].notna() &
                ja_tmp['start_date'].notna() &
                (ja_tmp['start_date'] < ja_tmp['prev_end']))
ja_issues['Overlapping periods'] = int(overlap_mask.sum())
if overlap_mask.any():
    print(f"[FOUND] Overlapping assignment periods: {overlap_mask.sum()} rows")
    print(ja_tmp[overlap_mask][['employee_id','start_date','end_date','prev_end']].head(3).to_string(index=False))
    print()
    ja.loc[overlap_mask[overlap_mask].index, 'start_date'] = ja_tmp.loc[overlap_mask,'prev_end'].values
    print(f"  Fix: adjusted start_date to align with previous end_date")


[FOUND] Overlapping assignment periods: 29 rows
 employee_id start_date   end_date   prev_end
EMP-11FA09BE 2024-07-29 2024-12-04 2024-08-25
EMP-13BD9955 2019-11-18        NaT 2019-12-04
EMP-21350978 2025-11-10        NaT 2025-11-24

  Fix: adjusted start_date to align with previous end_date


In [256]:
# ── Check 4: Spurious end_date on most recent assignment ─────────────────────
# Logic: the latest assignment per employee should be open (end_date = null).
# Except for Terminated or Retired employees.
# Cause: migration script incorrectly closed all records including current ones.
ja_s = ja.sort_values(['employee_id','start_date'])
is_latest = ja_s.groupby('employee_id')['start_date'].transform('max') == ja_s['start_date']

terminated_ids = set(
    emp.loc[emp['employment_status'].isin(['Terminated','Retired']), 'employee_id']
)
spurious_end = (
    is_latest &
    ja_s['end_date'].notna() &
    ~ja_s['employee_id'].isin(terminated_ids)
)
ja_issues['Spurious end_date on latest assignment'] = int(spurious_end.sum())
if spurious_end.any():
    print(f"[FOUND] Latest assignment with end_date: {spurious_end.sum()} rows")
    print(ja_s[spurious_end][['employee_id','start_date','end_date','job_level']].head(3).to_string(index=False))
    print()
ja.loc[spurious_end[spurious_end].index, 'end_date'] = None
print(f"  Fix: cleared end_date for {spurious_end.sum()} latest assignments")

[FOUND] Latest assignment with end_date: 68 rows
 employee_id start_date   end_date job_level
EMP-01A4023E 2023-03-20 2023-10-04        L5
EMP-01D9429F 2025-11-27 2026-02-06        L5
EMP-03EC2C2D 2025-07-12 2026-05-03        L3

  Fix: cleared end_date for 68 latest assignments


In [257]:
# ── Consistency: Terminated/Retired employees with open assignments ────────────
# Cause: HR system and job assignment module fell out of sync — status updated
# in employee master but the assignment was never closed.
open_for_leavers = (
    is_latest &
    ja_s['end_date'].isna() &
    ja_s['employee_id'].isin(terminated_ids)
)

if open_for_leavers.any():
    affected_term_dates = (
        ja_s[open_for_leavers][['employee_id']]
        .merge(emp[['employee_id','termination_date','term_date_imputed']], 
               on='employee_id', how='left')
    )
    ja.loc[open_for_leavers[open_for_leavers].index, 'end_date'] = \
        affected_term_dates['termination_date'].values
    ja.loc[open_for_leavers[open_for_leavers].index, 'end_date_imputed'] = \
        affected_term_dates['term_date_imputed'].values

print(f"[FOUND] Open assignments for Terminated/Retired: {open_for_leavers.sum()} — closed using termination_date")
imputed_count = affected_term_dates['term_date_imputed'].sum() if open_for_leavers.any() else 0
print(f"  - of which {imputed_count} used an imputed termination_date (flagged)")


[FOUND] Open assignments for Terminated/Retired: 1407 — closed using termination_date
  - of which 54 used an imputed termination_date (flagged)


In [258]:
# ── Derive promotion events and current flag on fully corrected data ──────────
ja = ja.sort_values(['employee_id','start_date'])
ja['prev_level']   = ja.groupby('employee_id')['job_level'].shift(1)
ja['is_promotion'] = ja['prev_level'].notna() & (ja['job_level'] > ja['prev_level'])
ja['is_current']   = ja['end_date'].isna()

print(f"\n=== Job assignments — summary ===")
for k, v in ja_issues.items():
    print(f"  {'FOUND':<6s} | {k:<45s} | {v:>5,} rows")
print(f"\nPromotion events detected: {ja['is_promotion'].sum():,}")
print(f"Open assignments: {ja['is_current'].sum():,}")


=== Job assignments — summary ===
  FOUND  | Self-referencing manager                      |    42 rows
  FOUND  | Missing manager_id                            |   475 rows
  FOUND  | Overlapping periods                           |    29 rows
  FOUND  | Spurious end_date on latest assignment        |    68 rows

Promotion events detected: 394
Open assignments: 3,796


### 3.3 Gigs, applications, skills, training, activity

In [259]:
valid_emp_ids = set(emp['employee_id'])

# ── Gigs ─────────────────────────────────────────────────────────────────────
gigs = raw['gig_master'].copy()

# Zero duration: vendor API defaults numeric fields to 0 when left blank
zero_dur = pd.to_numeric(gigs['duration_weeks_planned'], errors='coerce') == 0
print(f"[FOUND] Gigs with duration_weeks = 0: {zero_dur.sum()} — set to null")
gigs.loc[zero_dur, 'duration_weeks_planned'] = None

# Null hours: optional form field — keep null, don't impute
null_hrs = gigs['hours_per_week_planned'].isna()
print(f"[FOUND] Gigs with null hours_per_week: {null_hrs.sum()} — kept as null")

# Orphaned owner: employee no longer in the system (terminated and deleted)
orphan_owner = ~gigs['owner_employee_id'].isin(valid_emp_ids)
print(f"[FOUND] Gigs with orphaned owner: {orphan_owner.sum()} — flagged")
gigs['orphan_owner'] = orphan_owner
gigs['post_year']  = gigs['posted_date'].dt.year
gigs['post_month'] = gigs['posted_date'].dt.to_period('M').astype(str)


[FOUND] Gigs with duration_weeks = 0: 30 — set to null
[FOUND] Gigs with null hours_per_week: 37 — kept as null
[FOUND] Gigs with orphaned owner: 34 — flagged


In [260]:
# ── Applications ──────────────────────────────────────────────────────────────
apps = raw['applications'].copy()

# Zero assigned hours: numeric field left blank defaults to 0 in some form versions
# Treat as unknown rather than a genuine zero commitment
zero_hrs_app = apps['assigned_hours_per_week'].notna() & (apps['assigned_hours_per_week'] == 0)
print(f"[FOUND] Assigned hours = 0: {zero_hrs_app.sum()} rows — set to null")
apps.loc[zero_hrs_app, 'assigned_hours_per_week'] = None

# Duplicate applications: keep latest per employee+gig.
# Covers re-applications after withdrawal — we retain the most recent intent.
before = len(apps)
apps = apps.sort_values('application_date', ascending=False).drop_duplicates(
    subset=['employee_id','gig_id'], keep='first')
print(f"[FOUND] Duplicate employee+gig applications: {before - len(apps)} rows removed — kept latest")
apps['is_assigned'] = apps['application_status'] == 'Selected'

# Selected with no start date: status updated before assignment was confirmed
selected_no_start = (apps['application_status'] == 'Selected') & apps['assignment_start_date'].isna()
print(f"[FOUND] Selected with no assignment start date: {selected_no_start.sum()} — flagged")
apps['incomplete_assignment'] = selected_no_start
apps['is_assigned'] = apps['application_status'] == 'Selected'

print(f"\nFinal application rows: {len(apps):,}")


[FOUND] Assigned hours = 0: 18 rows — set to null
[FOUND] Duplicate employee+gig applications: 499 rows removed — kept latest
[FOUND] Selected with no assignment start date: 25 — flagged

Final application rows: 23,614


In [269]:
# ── Employee skills ───────────────────────────────────────────────────────────
skills_dim = raw['skill_dim'].copy()
emp_skills = raw['employee_skills'].copy()

# Out-of-range skill_level: import script used 0-based index instead of 1-5
emp_skills['skill_level'] = pd.to_numeric(emp_skills['skill_level'], errors='coerce')
out_of_range = ~emp_skills['skill_level'].between(1, 5)
print(f"[FOUND] skill_level out of range 1-5: {out_of_range.sum()} — clipped")
emp_skills['skill_level'] = emp_skills['skill_level'].clip(1, 5)

# Null skill_source: backfill job didn't cover all records when field was added
null_src = emp_skills['skill_source'].isna() | (emp_skills['skill_source'] == '')
print(f"[FOUND] Null skill_source: {null_src.sum()} — set to 'Unknown'")
emp_skills.loc[null_src, 'skill_source'] = 'Unknown'

# For each employee+skill, keep the record with the highest skill_level.
# Where multiple sources exist, flag it — the conflict itself is useful information.
# If levels are equal across sources, prefer the most recent assessment date.
emp_skills['has_multiple_sources'] = (
    emp_skills.groupby(['employee_id','skill_id'])['skill_source']
    .transform('nunique') > 1
)
emp_skills = (emp_skills
    .sort_values(['skill_level','added_date'], ascending=[False, False])
    .drop_duplicates(subset=['employee_id','skill_id'], keep='first'))
print(f"  After dedup: {len(emp_skills):,} employee-skill rows")

# Added_date before hire: skill imported from previous employer during onboarding
hire_dates = emp.set_index('employee_id')['hire_date']
emp_skills['_hire'] = emp_skills['employee_id'].map(hire_dates)
pre_hire = emp_skills['added_date'] < emp_skills['_hire']
print(f"[FOUND] Skill added before hire_date: {pre_hire.sum()} — flagged (legitimate historical skills)")
emp_skills['pre_hire_skill'] = pre_hire
emp_skills = emp_skills.drop(columns=['_hire'])


[FOUND] skill_level out of range 1-5: 1885 — clipped
[FOUND] Null skill_source: 2765 — set to 'Unknown'
  After dedup: 92,186 employee-skill rows
[FOUND] Skill added before hire_date: 1687 — flagged (legitimate historical skills)


In [270]:
# ── Training records ───────────────────────────────────────────────────────────
train_master = raw['training_master'].copy()
train_recs   = raw['training_records'].copy()
train_recs['hours'] = pd.to_numeric(train_recs['hours'], errors='coerce')

bad_hrs = (train_recs['hours'] == 0) | (train_recs['hours'] > 200)
print(f"\n[FOUND] Training hours = 0 or > 200: {bad_hrs.sum()} — set to null")
train_recs.loc[bad_hrs, 'hours'] = None

future_comp = train_recs['completion_date'] > REF_DATE
print(f"[FOUND] Completion date in future: {future_comp.sum()} — removed")
train_recs = train_recs[~future_comp]

dup_train = train_recs.duplicated(subset=['employee_id','training_id'], keep=False)
print(f"[FOUND] Duplicate training completions: {dup_train.sum()} — deduplicated")
train_recs = train_recs.drop_duplicates(subset=['employee_id','training_id'])


[FOUND] Training hours = 0 or > 200: 1264 — set to null
[FOUND] Completion date in future: 810 — removed
[FOUND] Duplicate training completions: 1652 — deduplicated


In [274]:
# ── Activity log ───────────────────────────────────────────────────────────────
VALID_EVENTS = {'Login','ViewGig','SearchGig','ApplyGig',
                'WithdrawApplication','UpdateProfile','AddSkill','CompleteGig'}
activity = raw['activity_log'].copy()

dup_events = activity.duplicated(subset=['event_id'], keep=False)
print(f"\n[FOUND] Duplicate event_ids: {dup_events.sum()} — deduplicated")
activity = activity.drop_duplicates(subset=['event_id'])

unknown_evt = ~activity['event_type'].isin(VALID_EVENTS)
print(f"[FOUND] Deprecated event types: {unknown_evt.sum()} — removed")
print(f"  Types: {activity.loc[unknown_evt,'event_type'].value_counts().to_dict()}")
activity = activity[~unknown_evt]

activity['event_month'] = activity['event_timestamp'].dt.to_period('M').astype(str)

print(f"\n=== Data quality pass complete ===")
print(f"Activity events after cleaning: {len(activity):,}")
print(f"Training records after cleaning: {len(train_recs):,}")
print(f"Employee skills after cleaning: {len(emp_skills):,}")


[FOUND] Duplicate event_ids: 10982 — deduplicated
[FOUND] Deprecated event types: 2734 — removed
  Types: {'RateGig': 485, 'ViewProfile': 474, 'ReportIssue': 455, 'InviteColleague': 450, 'ShareGig': 439, 'BookmarkGig': 431}

=== Data quality pass complete ===
Activity events after cleaning: 546,386
Training records after cleaning: 40,910
Employee skills after cleaning: 92,186
